In [ ]:
import pandas as pd
df = pd.read_csv("/content/noon_clean.csv")
print(df['sub_category'].unique())

['Dairy Cheese and Eggs' 'Tea' 'Dried Beans, Grains and Rice'
 'Pasta and Noodles' 'sauces' 'creal,oats,Jams&Honey' 'Herbs&spices'
 'juice' 'oils' 'salt' 'Soft drinks']


In [ ]:
import pandas as pd
df = pd.read_csv("/content/noon_clean.csv")

for cat in df['sub_category'].unique():
    print(f'\n=== {cat} ===')
    print(df[df['sub_category']==cat]['product_name'].head(5).tolist())


=== Dairy Cheese and Eggs ===
['مسحوق حليب كامل الدسم 15جرام عبوة من 20 قطعة', 'حليب مجفف كامل الدهن سريع الذوبان 700جرام', 'كرتون كامل الدسم عرض 1 لتر ايدج 6 عبوات', 'مجموعة حليب كامل الدسم 1.5لترات عبوة من 8 قطع', 'لبن السعودية – عبوة من 3 | لبن طازج ومغذي بطعم غني للاستخدام اليومي والشرب والطهي']

=== Tea ===
['ربيع الاقوي شاى اسود ناعم 100 جرام', 'احمد تى شاى اخضر بالنعناع اوراق 250 جرام', 'الشاى الاسود الانجليزى رقم 1من احمد تى 25 فتلة', 'احمد تى شاى انجليزى رقم 1 بدون علامة 100 كيس', 'أعشاب جوليا للتخسيس 10 كيس']

=== Dried Beans, Grains and Rice ===
['ارز وسكر وبهارات', 'ارز مصرى صبا 1 كيلو * 5', 'بوكس التوفير ( ريحانة أرز فاخر 1 كجم ريحانة أرز جاسمين 1 كجم ريحانة سكر 1 كجم ريحانة دقيق فاخر 1 كجم ريحانة فول تدميس بلدي 500 جم ريحانة فريك 500 جم ريحانة فلفل أحمر مطحون (بابريكا مدخن) 20 جم ريحانة فلفل أسود مطحون 20 جم ريحانة خلطة بهارات مطحونة 20 جم ريحانة شطة بلدى مطحونة 20 جم ريحانة قرفة مطحونة 20 جم ريحانة ملاحة توابل شرق اقصى ريحانة ملاحة مرقة فراخ ريحانة زيت دوار الشمس 0,7 لت

In [ ]:
!pip install deep-translator -q

import pandas as pd
from deep_translator import GoogleTranslator
import time

df = pd.read_csv("/content/noon_clean.csv")

def is_arabic(text):
    if pd.isna(text):
        return False
    return any('\u0600' <= c <= '\u06FF' for c in str(text))

# حدد الأسماء العربية
arabic_mask = df["product_name"].apply(is_arabic)
arabic_names = df[arabic_mask]["product_name"].unique().tolist()
print(f"Arabic names: {len(arabic_names)}")

# ترجم
translator = GoogleTranslator(source="ar", target="en")
translation_map = {}

for i, name in enumerate(arabic_names):
    try:
        translation_map[name] = translator.translate(name)
    except:
        translation_map[name] = name
    if (i+1) % 50 == 0:
        print(f"Progress: {i+1}/{len(arabic_names)}")
    time.sleep(0.1)

# طبق الترجمة
df["product_name"] = df["product_name"].apply(lambda x: translation_map.get(x, x))

# Save
df.to_csv("/content/noon_translated.csv", index=False)
print("✅ Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00
Arabic names: 822
Progress: 50/822
Progress: 100/822
Progress: 150/822
Progress: 200/822
Progress: 250/822
Progress: 300/822
Progress: 350/822
Progress: 400/822
Progress: 450/822
Progress: 500/822
Progress: 550/822
Progress: 600/822
Progress: 650/822
Progress: 700/822
Progress: 750/822
Progress: 800/822
✅ Done!


In [ ]:
import pandas as pd

df = pd.read_csv("/content/noon_translated.csv")

# Check remaining Arabic
arabic_mask = df["product_name"].apply(lambda x: any('\u0600' <= c <= '\u06FF' for c in str(x)) if pd.notna(x) else False)
print(f"Remaining Arabic: {arabic_mask.sum()}")
print()
print("Sample translations:")
print(df["product_name"].head(10).tolist())

Remaining Arabic: 0

Sample translations:
['Full cream milk powder 15 grams, pack of 20 pieces', 'Instant full fat milk powder, 700 grams', 'Full-fat carton, 1 liter width, 6 packs', 'Whole milk set 1.5 liters, pack of 8 pieces', 'Saudia Yoghurt – Pack of 3 | Fresh and nutritious milk with a rich taste for daily use, drinking and cooking', 'Full cream evaporated milk 410 g', 'Professional coconut milk 1 litre', 'Full cream milk, 1 liter, pack of 12 pieces', 'Coconut milk N&G', 'Long Life Skim Milk, 1 Liter, Pack of 12']


In [ ]:
import pandas as pd
import re

df = pd.read_csv("/content/noon_translated.csv")

def get_product_type_noon(product_name):
    if pd.isna(product_name):
        return "Unknown"

    name = str(product_name).lower()

    # Dairy
    if any(k in name for k in ["milk", "cream", "evaporated", "condensed", "powder milk"]):
        return "Milk"
    if any(k in name for k in ["cheese", "picon", "gouda", "cheddar"]):
        return "Cheese"
    if any(k in name for k in ["egg", "eggs"]):
        return "Eggs"
    if any(k in name for k in ["yogurt", "yoghurt", "laban"]):
        return "Yogurt"
    if any(k in name for k in ["butter"]):
        return "Butter"

    # Grains
    if any(k in name for k in ["rice", "basmati"]):
        return "Rice"
    if any(k in name for k in ["lentil", "lentils", "adas"]):
        return "Lentils"
    if any(k in name for k in ["bean", "beans", "fava", "foul"]):
        return "Beans"
    if any(k in name for k in ["wheat", "freekeh", "bulgur", "couscous"]):
        return "Grains"

    # Pasta
    if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni",
                                "fusilli", "lasagna", "fettuc", "risoni"]):
        return "Pasta"
    if any(k in name for k in ["noodle", "ramen", "indomie"]):
        return "Noodles"
    if any(k in name for k in ["vermicelli"]):
        return "Vermicelli"

    # Cereal
    if any(k in name for k in ["oat", "oats"]):
        return "Oats"
    if any(k in name for k in ["cereal", "cornflakes", "granola", "muesli"]):
        return "Cereal"

    # Jams & Spreads
    if any(k in name for k in ["honey", "عسل"]):
        return "Honey"
    if any(k in name for k in ["jam", "jelly"]):
        return "Jams"
    if any(k in name for k in ["spread", "nutella", "peanut"]):
        return "Spreads"

    # Herbs & Spices
    if any(k in name for k in ["spice", "pepper", "cumin", "cinnamon",
                                "turmeric", "seasoning", "herb", "thyme",
                                "ginger", "rosemary", "basil"]):
        return "Spices"

    # Oils & Sauces
    if any(k in name for k in ["oil", "vinegar"]):
        return "Oils"
    if any(k in name for k in ["sauce", "ketchup", "mayo", "mustard", "paste"]):
        return "Sauces"

    # Drinks
    if any(k in name for k in ["tea", "green tea", "herbal"]):
        return "Tea"
    if any(k in name for k in ["juice", "nectar"]):
        return "Juices"
    if any(k in name for k in ["water"]):
        return "Water"
    if any(k in name for k in ["coffee", "espresso", "cappuccino"]):
        return "Coffee"
    if any(k in name for k in ["soda", "cola", "pepsi", "sprite",
                                "soft drink", "energy drink", "carbonated"]):
        return "Soft Drinks"

    # Salt & Sugar
    if any(k in name for k in ["salt"]):
        return "Salt"
    if any(k in name for k in ["sugar", "sweetener", "stevia"]):
        return "Sugar"

    return "Other"

df["product_type"] = df["product_name"].apply(get_product_type_noon)

print("Product types:")
print(df["product_type"].value_counts())
print()
print(f"Unknown/Other: {(df['product_type'] == 'Other').sum()}")

Product types:
product_type
Other          604
Spices         520
Oils           321
Sauces         294
Tea            250
Honey          182
Milk           169
Rice           147
Sugar          141
Pasta          117
Spreads         79
Juices          74
Jams            71
Soft Drinks     58
Butter          48
Noodles         46
Oats            46
Beans           36
Cereal          31
Salt            26
Cheese          23
Lentils         20
Grains          10
Eggs             6
Vermicelli       5
Water            3
Coffee           2
Name: count, dtype: int64

Unknown/Other: 604


In [ ]:
import pandas as pd

df = pd.read_csv("/content/noon_translated.csv")

# شوف الـ Other products
other_mask = df["product_name"].apply(lambda x: True)  # temp

def get_product_type_noon(product_name):
    if pd.isna(product_name):
        return "Unknown"
    name = str(product_name).lower()
    if any(k in name for k in ["milk", "cream", "evaporated", "condensed"]):
        return "Milk"
    if any(k in name for k in ["cheese", "picon", "gouda", "cheddar"]):
        return "Cheese"
    if any(k in name for k in ["egg", "eggs"]):
        return "Eggs"
    if any(k in name for k in ["yogurt", "yoghurt", "laban"]):
        return "Yogurt"
    if any(k in name for k in ["butter"]):
        return "Butter"
    if any(k in name for k in ["rice", "basmati"]):
        return "Rice"
    if any(k in name for k in ["lentil", "lentils"]):
        return "Lentils"
    if any(k in name for k in ["bean", "beans", "fava"]):
        return "Beans"
    if any(k in name for k in ["wheat", "freekeh", "bulgur", "couscous"]):
        return "Grains"
    if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni","fusilli", "lasagna", "fettuc", "risoni"]):
        return "Pasta"
    if any(k in name for k in ["noodle", "ramen", "indomie"]):
        return "Noodles"
    if any(k in name for k in ["vermicelli"]):
        return "Vermicelli"
    if any(k in name for k in ["oat", "oats"]):
        return "Oats"
    if any(k in name for k in ["cereal", "cornflakes", "granola", "muesli"]):
        return "Cereal"
    if any(k in name for k in ["honey"]):
        return "Honey"
    if any(k in name for k in ["jam", "jelly"]):
        return "Jams"
    if any(k in name for k in ["spread", "nutella", "peanut"]):
        return "Spreads"
    if any(k in name for k in ["spice", "pepper", "cumin", "cinnamon","turmeric", "seasoning", "herb", "thyme","ginger", "rosemary", "basil"]):
        return "Spices"
    if any(k in name for k in ["oil", "vinegar"]):
        return "Oils"
    if any(k in name for k in ["sauce", "ketchup", "mayo", "mustard", "paste"]):
        return "Sauces"
    if any(k in name for k in ["tea", "green tea", "herbal"]):
        return "Tea"
    if any(k in name for k in ["juice", "nectar"]):
        return "Juices"
    if any(k in name for k in ["water"]):
        return "Water"
    if any(k in name for k in ["coffee", "espresso", "cappuccino"]):
        return "Coffee"
    if any(k in name for k in ["soda", "cola", "pepsi", "sprite","soft drink", "energy drink", "carbonated"]):
        return "Soft Drinks"
    if any(k in name for k in ["salt"]):
        return "Salt"
    if any(k in name for k in ["sugar", "sweetener", "stevia"]):
        return "Sugar"
    return "Other"

df["product_type"] = df["product_name"].apply(get_product_type_noon)

# شوف الـ Other
others = df[df["product_type"] == "Other"]["product_name"].head(30).tolist()
for p in others:
    print(p)

Full-fat carton, 1 liter width, 6 packs
Trocopapa pomegranate drink - 300g
Abu Qus Cardamom Flavored Drink, 240 ml | A refreshing drink with the authentic taste of Arabic cardamom
Ahmed T. Earl Gray 100 filament
With saffron, a package of 10 envelopes, 200 grams
Rosehip, hibiscus and cherry 20 wafers
10 sachets of Baby Calming Drink, 5g each (Pack of 2)
Lipton Phoenix Hibiscus 20 strands
Lipton Phoenix Mint 20 wicks
Guava Leaves Drink 20 Sachets
Fennel Drink Pack of 20
They forget 100 bags
Chamomile from 12 filtered bags
Sage Drink 20 Sachets
Samyang Nolds Korean Instant Chicken Habanero Lemon 3*140g
Samyang Nolds Korean Instant Chicken Habanero Lemon 5*140g
Samyang Nolds Instant Korean Spicy Chicken Habanero Lemon 4*140 grams
Saming Nolds Korean Instant Pot Spicy Chicken Flavor Original Taste 140 Grams *3
Saming Nolds Korean Instant Pot Spicy Chicken Flavor Original Taste 140 Grams *5
Farfalli Tund 500g
Lupine seeds
Chickpeas 500 grams
Chickpeas 250 grams
Organic Cocoa Powder, 8.5 oz 

In [ ]:
import pandas as pd

df = pd.read_csv("/content/noon_translated.csv")

def get_product_type_noon(product_name):
    if pd.isna(product_name):
        return "Unknown"
    name = str(product_name).lower()

    # Milk & Dairy
    if any(k in name for k in ["milk", "evaporated", "condensed"]):
        return "Milk"
    if any(k in name for k in ["cheese", "picon", "gouda", "cheddar"]):
        return "Cheese"
    if any(k in name for k in ["egg", "eggs"]):
        return "Eggs"
    if any(k in name for k in ["yogurt", "yoghurt", "laban"]):
        return "Yogurt"
    if any(k in name for k in ["butter"]):
        return "Butter"
    if any(k in name for k in ["cream"]):
        return "Milk"

    # Grains
    if any(k in name for k in ["rice", "basmati"]):
        return "Rice"
    if any(k in name for k in ["lentil", "lentils"]):
        return "Lentils"
    if any(k in name for k in ["bean", "beans", "fava", "lupine", "chickpea"]):
        return "Beans"
    if any(k in name for k in ["wheat", "freekeh", "bulgur", "couscous"]):
        return "Grains"

    # Pasta & Noodles
    if any(k in name for k in ["noodle", "ramen", "indomie", "samyang",
                                "saming", "nolds", "instant pot"]):
        return "Noodles"
    if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni",
                                "fusilli", "lasagna", "fettuc", "risoni",
                                "farfalli", "rings"]):
        return "Pasta"
    if any(k in name for k in ["vermicelli"]):
        return "Vermicelli"

    # Cereal
    if any(k in name for k in ["oat", "oats"]):
        return "Oats"
    if any(k in name for k in ["cereal", "cornflakes", "granola", "muesli"]):
        return "Cereal"

    # Jams & Spreads
    if any(k in name for k in ["honey"]):
        return "Honey"
    if any(k in name for k in ["jam", "jelly"]):
        return "Jams"
    if any(k in name for k in ["spread", "nutella", "peanut", "cocoa", "chocolate"]):
        return "Spreads"

    # Herbs & Spices
    if any(k in name for k in ["spice", "pepper", "cumin", "cinnamon",
                                "turmeric", "seasoning", "thyme",
                                "ginger", "rosemary", "basil"]):
        return "Spices"

    # Oils & Sauces
    if any(k in name for k in ["oil", "vinegar"]):
        return "Oils"
    if any(k in name for k in ["sauce", "ketchup", "mayo", "mustard", "paste"]):
        return "Sauces"

    # Drinks
    if any(k in name for k in ["tea", "earl gray", "lipton", "hibiscus",
                                "chamomile", "sage", "fennel", "rosehip",
                                "herbal", "herb"]):
        return "Tea"
    if any(k in name for k in ["drink", "beverage", "cardamom", "pomegranate",
                                "guava", "calming", "saffron"]):
        return "Tea"
    if any(k in name for k in ["juice", "nectar"]):
        return "Juices"
    if any(k in name for k in ["water"]):
        return "Water"
    if any(k in name for k in ["coffee", "espresso", "cappuccino", "nescafe"]):
        return "Coffee"
    if any(k in name for k in ["soda", "cola", "pepsi", "sprite",
                                "soft drink", "energy drink", "carbonated"]):
        return "Soft Drinks"

    # Salt & Sugar
    if any(k in name for k in ["salt"]):
        return "Salt"
    if any(k in name for k in ["sugar", "sweetener", "stevia", "fructose"]):
        return "Sugar"

    return "Other"

df["product_type"] = df["product_name"].apply(get_product_type_noon)

print("Product types:")
print(df["product_type"].value_counts())
print()
print(f"Other count: {(df['product_type'] == 'Other').sum()}")
print()
print("Sample Others:")
print(df[df["product_type"] == "Other"]["product_name"].head(20).tolist())

Product types:
product_type
Other          499
Spices         471
Tea            419
Oils           327
Sauces         289
Honey          180
Milk           156
Rice           147
Sugar          141
Pasta          126
Spreads        107
Jams            71
Noodles         62
Butter          58
Juices          57
Beans           45
Oats            43
Cereal          31
Salt            26
Cheese          25
Lentils         20
Grains          10
Eggs             6
Vermicelli       5
Soft Drinks      3
Water            3
Yogurt           1
Coffee           1
Name: count, dtype: int64

Other count: 499

Sample Others:
['Full-fat carton, 1 liter width, 6 packs', 'They forget 100 bags', 'Family-sized quick-prepared koshari meal, 530 grams', 'Premium orzo 400 grams', 'Instant koshari 188 grams', 'Egyptian popia 500 grams', 'Ribal Sobia Powder - 600 grams', 'Hummus Al-Sham', 'Grits - 500 grams', 'hummus', 'Premium mint filter 20', 'black eyed peas', 'Canned thermos 500gm', 'Ubuki Halal Cup Tobuk

In [ ]:
# Pasta (orzo, koshari)
if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni",
                            "fusilli", "lasagna", "fettuc", "risoni",
                            "farfalli", "rings", "orzo", "koshari",
                            "popia", "grits"]):
    return "Pasta"

# Noodles (tobuki)
if any(k in name for k in ["noodle", "ramen", "indomie", "samyang",
                            "saming", "nolds", "tobuki", "ubuki",
                            "yubuki", "instant pot", "cup"]):
    return "Noodles"

# Beans (hummus, black eyed peas, thermos, sobia)
if any(k in name for k in ["bean", "beans", "fava", "lupine", "chickpea",
                            "hummus", "black eyed", "thermos", "sobia"]):
    return "Beans"

# Tea (mint, forget = يُنسى tea brand?)
if any(k in name for k in ["tea", "earl gray", "lipton", "hibiscus",
                            "chamomile", "sage", "fennel", "rosehip",
                            "herbal", "herb", "mint filter", "forget"]):
    return "Tea"

SyntaxError: 'return' outside function (1145380754.py, line 6)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/noon_translated.csv")

def get_product_type_noon(product_name):
    if pd.isna(product_name):
        return "Unknown"
    name = str(product_name).lower()

    # Milk & Dairy
    if any(k in name for k in ["milk", "evaporated", "condensed", "carton"]):
        return "Milk"
    if any(k in name for k in ["cheese", "picon", "gouda", "cheddar"]):
        return "Cheese"
    if any(k in name for k in ["egg", "eggs"]):
        return "Eggs"
    if any(k in name for k in ["yogurt", "yoghurt", "laban"]):
        return "Yogurt"
    if any(k in name for k in ["butter"]):
        return "Butter"
    if any(k in name for k in ["cream"]):
        return "Milk"

    # Grains
    if any(k in name for k in ["rice", "basmati"]):
        return "Rice"
    if any(k in name for k in ["lentil", "lentils"]):
        return "Lentils"
    if any(k in name for k in ["bean", "beans", "fava", "lupine",
                                "chickpea", "hummus", "black eyed",
                                "thermos", "sobia"]):
        return "Beans"
    if any(k in name for k in ["wheat", "freekeh", "bulgur", "couscous"]):
        return "Grains"

    # Noodles
    if any(k in name for k in ["noodle", "ramen", "indomie", "samyang",
                                "saming", "nolds", "tobuki", "ubuki",
                                "yubuki", "instant pot"]):
        return "Noodles"

    # Pasta
    if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni",
                                "fusilli", "lasagna", "fettuc", "risoni",
                                "farfalli", "rings", "orzo", "koshari",
                                "popia", "grits"]):
        return "Pasta"
    if any(k in name for k in ["vermicelli"]):
        return "Vermicelli"

    # Cereal
    if any(k in name for k in ["oat", "oats"]):
        return "Oats"
    if any(k in name for k in ["cereal", "cornflakes", "granola", "muesli"]):
        return "Cereal"

    # Jams & Spreads
    if any(k in name for k in ["honey"]):
        return "Honey"
    if any(k in name for k in ["jam", "jelly"]):
        return "Jams"
    if any(k in name for k in ["spread", "nutella", "peanut", "cocoa", "chocolate"]):
        return "Spreads"

    # Herbs & Spices
    if any(k in name for k in ["spice", "pepper", "cumin", "cinnamon",
                                "turmeric", "seasoning", "thyme",
                                "ginger", "rosemary", "basil"]):
        return "Spices"

    # Oils & Sauces
    if any(k in name for k in ["oil", "vinegar"]):
        return "Oils"
    if any(k in name for k in ["sauce", "ketchup", "mayo", "mustard", "paste"]):
        return "Sauces"

    # Tea & Drinks
    if any(k in name for k in ["tea", "earl gray", "lipton", "hibiscus",
                                "chamomile", "sage", "fennel", "rosehip",
                                "herbal", "mint filter", "forget"]):
        return "Tea"
    if any(k in name for k in ["drink", "beverage", "cardamom", "pomegranate",
                                "guava", "calming", "saffron"]):
        return "Tea"
    if any(k in name for k in ["juice", "nectar"]):
        return "Juices"
    if any(k in name for k in ["water"]):
        return "Water"
    if any(k in name for k in ["coffee", "espresso", "cappuccino", "nescafe"]):
        return "Coffee"
    if any(k in name for k in ["soda", "cola", "pepsi", "sprite",
                                "soft drink", "energy drink", "carbonated"]):
        return "Soft Drinks"

    # Salt & Sugar
    if any(k in name for k in ["salt"]):
        return "Salt"
    if any(k in name for k in ["sugar", "sweetener", "stevia", "fructose"]):
        return "Sugar"

    return "Other"

df["product_type"] = df["product_name"].apply(get_product_type_noon)

print("Product types:")
print(df["product_type"].value_counts())
print()
print(f"Other count: {(df['product_type'] == 'Other').sum()}")
print()
print("Sample Others:")
print(df[df["product_type"] == "Other"]["product_name"].head(20).tolist())

Product types:
product_type
Other          484
Spices         470
Tea            412
Oils           326
Sauces         287
Honey          180
Milk           160
Rice           147
Sugar          142
Pasta          131
Spreads        107
Jams            71
Noodles         68
Butter          58
Juices          57
Beans           55
Oats            43
Cereal          31
Salt            26
Cheese          25
Lentils         20
Grains          10
Eggs             6
Vermicelli       5
Soft Drinks      3
Water            3
Yogurt           1
Coffee           1
Name: count, dtype: int64

Other count: 484

Sample Others:
['Julia herbs for weight loss 10 sachets', 'Hot Bustard, Great Garlic, Kissed - 250-270-270 gm', 'Premium Ramadan Food Essentials Bundle - Comprehensive Household Grocery Box', "Bob's Red Mill Raw Whole Brown Flaxseed, 13-ounce", 'protein bar Hazelnut Bar 70grams', 'said Korean Red Ginseng Golden Liquid 30g', 'Dubai filling Kunafa & Pistachio 500 Gram', 'Traditional Musli 350gr

In [ ]:
import pandas as pd

# شيل الـ Other
df_clean = df[df["product_type"] != "Other"].reset_index(drop=True)
print(f"After removing Other: {df_clean.shape}")

# Rename start_date → date
df_clean = df_clean.rename(columns={"start_date": "date"})

# Final column order زي أمازون بالظبط
final_cols = [
    "sku", "source", "category", "sub_category", "product_type",
    "product_name", "brand", "price", "price_per_unit",
    "number_of_items", "item_weight_grams", "date"
]
df_clean = df_clean[[c for c in final_cols if c in df_clean.columns]]

print(f"Columns: {df_clean.columns.tolist()}")
print(f"Sample:")
print(df_clean.head(3).to_string())

# Save
df_clean.to_csv("/content/noon_final.csv", index=False)

from google.colab import files
files.download("/content/noon_final.csv")
print("✅ Done!")

After removing Other: (2845, 14)
Columns: ['sku', 'source', 'category', 'sub_category', 'product_type', 'product_name', 'brand', 'price', 'price_per_unit', 'number_of_items', 'item_weight_grams', 'date']
Sample:
                      sku source category           sub_category product_type                                        product_name  brand   price  price_per_unit  number_of_items  item_weight_grams        date
0              N28853935A   noon  Grocery  Dairy Cheese and Eggs         Milk  Full cream milk powder 15 grams, pack of 20 pieces   ميرو  147.25          7.3625             20.0               15.0  2026-05-25
1              N28853944A   noon  Grocery  Dairy Cheese and Eggs         Milk             Instant full fat milk powder, 700 grams  تامبو  191.75         95.8750              2.0              700.0  2026-05-25
2  Z9D48D86C44595540A820Z   noon  Grocery  Dairy Cheese and Eggs         Milk             Full-fat carton, 1 liter width, 6 packs  جهينه  274.95         45.8250 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done!


In [ ]:
!pip install deep-translator -q

import pandas as pd
from deep_translator import GoogleTranslator
import time

df = pd.read_csv("/content/noon_final.csv")

def is_arabic(text):
    if pd.isna(text):
        return False
    return any('\u0600' <= c <= '\u06FF' for c in str(text))

# حدد الـ brands العربية
arabic_brands = df[df["brand"].apply(is_arabic)]["brand"].unique().tolist()
print(f"Arabic brands: {len(arabic_brands)}")

# ترجم
translator = GoogleTranslator(source="ar", target="en")
translation_map = {}

for i, brand in enumerate(arabic_brands):
    try:
        translation_map[brand] = translator.translate(brand)
    except:
        translation_map[brand] = brand
    time.sleep(0.1)

# طبق الترجمة
df["brand"] = df["brand"].apply(lambda x: translation_map.get(x, x))

print("Sample brand translations:")
for ar, en in list(translation_map.items())[:10]:
    print(f"  {ar} → {en}")

# Save
df.to_csv("/content/noon_final.csv", index=False)

from google.colab import files
files.download("/content/noon_final.csv")
print("✅ Done!")

Arabic brands: 186
Sample brand translations:
  ميرو → Miro
  تامبو → Tambo
  جهينه → Juhayna
  السعودية → Saudi Arabia
  بوني → Bonnie
  ألبرو → Alpro
  ذا ميلك مان → The Milk Man
  نيدو → Nido
  جهينة → Juhayna
  المراعي → Pastures


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done!


In [ ]:
import pandas as pd

# Load
amazon = pd.read_csv("/content/amazon_clean.csv")
noon = pd.read_csv("/content/noon_final.csv")

print(f"Amazon: {amazon.shape}")
print(f"Noon: {noon.shape}")

# Combine
combined = pd.concat([amazon, noon], ignore_index=True)
print(f"Combined: {combined.shape}")

# Check
print(f"\nSource counts:")
print(combined["source"].value_counts())
print(f"\nProduct types:")
print(combined["product_type"].value_counts())
print(f"\nNulls:")
print(combined.isnull().sum())
print(f"\nSample:")
print(combined.head(3).to_string())

# Save
combined.to_csv("/content/unified_prices.csv", index=False)
combined.to_parquet("/content/unified_prices.parquet", index=False, engine="pyarrow")

from google.colab import files
files.download("/content/unified_prices.csv")
files.download("/content/unified_prices.parquet")

print("\n✅ Done!")

Amazon: (11039, 12)
Noon: (2845, 12)
Combined: (13884, 12)

Source counts:
source
amazon    11039
noon       2845
Name: count, dtype: int64

Product types:
product_type
Cooking & Baking    2280
Spices              2221
Coffee              2088
Tea                 1546
Sauces              1063
Jams                 532
Spreads              528
Juices               446
Pasta                423
Rice                 376
Oils                 326
Rice & Pasta         299
Milk                 251
Honey                190
Noodles              185
Soft Drinks          177
Oats                 151
Sugar                142
Cereal               137
Cereal & Oats        135
Water                100
Herbs                 61
Butter                58
Beans                 55
Vermicelli            26
Salt                  26
Cheese                25
Lentils               20
Grains                10
Eggs                   6
Yogurt                 1
Name: count, dtype: int64

Nulls:
sku                   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done!


In [ ]:
print(combined[combined["product_type"] == "Rice & Pasta"]["product_name"].head(10).tolist())
print()
print(combined[combined["product_type"] == "Cereal & Oats"]["product_name"].head(10).tolist())
print()
print(combined[combined["product_type"] == "Cooking & Baking"]["product_name"].head(10).tolist())

['Italiano serpentini 400g', 'Al Doha Freekeh - 500 grams', 'El maleka big rings - 1 kg', 'El maleka elbow 400gm', 'Zaeem Popcorn - 500g', 'El maleka big rings 400gm', 'Kayy', 'Afandy Afandi Brown Lentils 250g', 'Sedr golden basmati rice 1 kg', 'Eco Healthy - Whole Grain Oats - 1 kg']

['Organic Nation', 'Scrunch - belgian dark chocolate barks with almond 35g', "Temmy's Choco Scoops Puffed Wheat with Cocoa - 500 Grams", 'Scrunch - almond, cashew and cranberry clusters 40g', 'Al Doha Egyptian Yellow Lentil-500 grams', 'Scrunch - cashew, almond and pumpkin seed clusters 40g', 'Rehana yellow lentils - 500 gm', 'Alsuhagy corn, 500 g', 'Imtenan Cornflex Grateful 13.5 oz', 'Rehana popcorn corn - 500 gm']

['Crystal Sunflower Oil, 2.2 Liter', 'Dobella whole wheat diet flour, 1 kg', 'Afia sunflower oil, 2.2 liters', 'Aldoha White Sugar, 1 KG', 'Al Doha Egyptian Flour, 1 kg (Pack of 1)', 'Rehana natural vinegar - 1 l', 'zamzam sugar 1 kg', 'Organo Natural Vinegar With 5% Acidity 900 ML - Clear'

In [ ]:
import pandas as pd

df = pd.read_csv("/content/unified_prices.csv")

def fix_product_type(row):
    name = str(row["product_name"]).lower()
    ptype = str(row["product_type"])

    # لو product_type محتاج تصليح
    if ptype in ["Rice & Pasta", "Cereal & Oats", "Cooking & Baking",
                 "Herbs & Spices", "Jams, Honey & Spreads",
                 "Sauces, Gravies & Marinades"]:

        # Rice & Pasta
        if any(k in name for k in ["vermicelli"]):
            return "Vermicelli"
        if any(k in name for k in ["noodle", "ramen", "indomie"]):
            return "Noodles"
        if any(k in name for k in ["rice", "basmati"]):
            return "Rice"
        if any(k in name for k in ["pasta", "spaghetti", "penne", "macaroni",
                                    "fusilli", "lasagna", "fettuc", "risoni",
                                    "serpentini", "elbow", "rings", "farfalli"]):
            return "Pasta"
        if any(k in name for k in ["freekeh", "bulgur", "wheat", "couscous"]):
            return "Grains"
        if any(k in name for k in ["lentil", "lentils"]):
            return "Lentils"
        if any(k in name for k in ["bean", "beans", "fava", "chickpea"]):
            return "Beans"
        if any(k in name for k in ["popcorn", "corn"]):
            return "Cereal"

        # Cereal & Oats
        if any(k in name for k in ["oat", "oats"]):
            return "Oats"
        if any(k in name for k in ["cereal", "cornflakes", "granola",
                                    "muesli", "chocolate", "scoops", "puffed"]):
            return "Cereal"

        # Cooking & Baking
        if any(k in name for k in ["oil", "sunflower", "olive", "corn oil"]):
            return "Oils"
        if any(k in name for k in ["flour", "pancake", "cake flour"]):
            return "Flour"
        if any(k in name for k in ["sugar"]):
            return "Sugar"
        if any(k in name for k in ["vinegar"]):
            return "Oils"
        if any(k in name for k in ["salt"]):
            return "Salt"

        # Herbs & Spices
        if any(k in name for k in ["spice", "pepper", "cumin", "cinnamon",
                                    "turmeric", "seasoning", "herb", "thyme",
                                    "ginger", "rosemary", "basil"]):
            return "Spices"

        # Jams & Spreads
        if any(k in name for k in ["honey"]):
            return "Honey"
        if any(k in name for k in ["jam", "jelly"]):
            return "Jams"
        if any(k in name for k in ["spread", "nutella", "peanut"]):
            return "Spreads"

        # Sauces
        if any(k in name for k in ["sauce", "ketchup", "mayo", "mustard", "paste"]):
            return "Sauces"

        return "Other"

    return ptype

df["product_type"] = df.apply(fix_product_type, axis=1)

print("Product types after fix:")
print(df["product_type"].value_counts())
print(f"\nOther count: {(df['product_type'] == 'Other').sum()}")

# Fix brand nulls
df["brand"] = df["brand"].fillna("Unknown")

# Save
df.to_csv("/content/unified_prices.csv", index=False)
df.to_parquet("/content/unified_prices.parquet", index=False, engine="pyarrow")

from google.colab import files
files.download("/content/unified_prices.csv")
files.download("/content/unified_prices.parquet")

print("✅ Done!")

Product types after fix:
product_type
Spices         2279
Coffee         2088
Tea            1546
Sauces         1104
Other          1047
Oils            740
Honey           647
Jams            550
Spreads         530
Pasta           450
Juices          446
Sugar           414
Rice            404
Cereal          271
Milk            251
Noodles         198
Soft Drinks     177
Oats            172
Water           100
Beans            96
Flour            87
Herbs            61
Butter           58
Lentils          45
Grains           35
Salt             30
Vermicelli       26
Cheese           25
Eggs              6
Yogurt            1
Name: count, dtype: int64

Other count: 1047


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done!


In [ ]:
import pandas as pd

df = pd.read_csv("/content/unified_prices.csv")

print("Sample Others:")
others = df[df["product_type"] == "Other"]["product_name"].head(30).tolist()
for p in others:
    print(p)

Sample Others:
Kayy
Abu Auf White Quinoa, 400 gm
Al Doha Black Eyed Peas - 500 grams
Tik
ONIC NATURALLY YOURS - Quinoa Seeds - 250g
Granoro
Tik
De Cecco De Cecco Potato Gnocchi 500G
Granoro
Barilla Pesti Alla Genovese 190g
Italiano Big Shell - 250 Grams
Biohayah White Quinoa Seeds - Raw Organic & Non-GMO - Gluten-Free, Vegan Superfood - Rich in Plant Protein & Fiber - Healthy Cooking & Daily Nutrition - 200g
Best Fighter To Kill Bed Bugs Completely
Granoro
Topokki Cup Cheese, Yopokki, 113g
Abu Auf TriColor Quinoa, 400 gm
3 Boxes - Crackers Prawn Flavor - 200 gm
Granoro
Granoro
California Garden Canned Chick Peas Ready To Eat 400g
Alsuhagy chikpeas, 500 g
Al-Abd Mixed Kahk Box, 116 Pieces, 1 Kilo
Hawa Crowna 400g (5 pieces, slice)
Biohayah White Quinoa Seeds - Raw Organic & Non-GMO - Gluten-Free, Vegan Superfood - Rich in Plant Protein & Fiber - Healthy Cooking & Daily Nutrition - 400g
Eid kahk from Eatool, mixed box (1 kg)
A box of mixed Eid cake from Al-Abd (2 kg)
zamzam humous 500 gm

In [ ]:
import pandas as pd

df = pd.read_csv("/content/unified_prices.csv")

def fix_other(row):
    name = str(row["product_name"]).lower()
    ptype = str(row["product_type"])

    if ptype != "Other":
        return ptype

    # Beans & Legumes
    if any(k in name for k in ["quinoa", "black eyed", "chickpea", "chickpeas",
                                "hummus", "humous", "chick peas"]):
        return "Beans"

    # Pasta
    if any(k in name for k in ["gnocchi", "shell", "crowna", "crackers",
                                "pesti", "pesto"]):
        return "Pasta"

    # Noodles
    if any(k in name for k in ["topokki", "yopokki", "tteok"]):
        return "Noodles"

    # Cooking & Baking
    if any(k in name for k in ["baking powder", "vanilla", "yeast"]):
        return "Flour"

    # Snacks → شيلهم
    if any(k in name for k in ["kahk", "biscuit", "cake", "prawn",
                                "snack", "cookie", "wafer"]):
        return "Remove"

    # Non-food → شيلهم
    if any(k in name for k in ["bed bug", "insect", "kill"]):
        return "Remove"

    # Brand names بس من غير وصف (اسم قصير جداً) → شيلهم
    if len(name.strip()) <= 5:
        return "Remove"

    return "Other"

df["product_type"] = df.apply(fix_other, axis=1)

# شيل الـ Remove والـ Other
before = len(df)
df = df[~df["product_type"].isin(["Remove", "Other"])].reset_index(drop=True)
print(f"Removed: {before - len(df)} rows")
print(f"Final shape: {df.shape}")
print()
print("Product types:")
print(df["product_type"].value_counts())

# Save
df.to_csv("/content/unified_prices_final.csv", index=False)
df.to_parquet("/content/unified_prices_final.parquet", index=False, engine="pyarrow")

from google.colab import files
files.download("/content/unified_prices_final.csv")
files.download("/content/unified_prices_final.parquet")

print("\n✅ Done!")

Removed: 970 rows
Final shape: (12914, 12)

Product types:
product_type
Spices         2279
Coffee         2088
Tea            1546
Sauces         1104
Oils            740
Honey           647
Jams            550
Spreads         530
Pasta           464
Juices          446
Sugar           414
Rice            404
Cereal          271
Milk            251
Noodles         199
Soft Drinks     177
Oats            172
Flour           136
Beans           109
Water           100
Herbs            61
Butter           58
Lentils          45
Grains           35
Salt             30
Vermicelli       26
Cheese           25
Eggs              6
Yogurt            1
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done!


In [ ]:
import pandas as pd

df = pd.read_csv("/content/unified_prices_final.csv")

print("=== Basic Info ===")
print(f"Shape: {df.shape}")
print()

print("=== Nulls ===")
print(df.isnull().sum())
print()

print("=== Price Stats ===")
print(df["price"].describe())
print()

print("=== price_per_unit Stats ===")
print(df["price_per_unit"].describe())
print()

print("=== Date ===")
print(df["date"].unique())
print()

print("=== Sample ===")
print(df.head(5).to_string())

=== Basic Info ===
Shape: (12914, 12)

=== Nulls ===
sku                     0
source                  0
category                0
sub_category            0
product_type            0
product_name            0
brand                   0
price                   0
price_per_unit          0
number_of_items      4559
item_weight_grams    4694
date                    0
dtype: int64

=== Price Stats ===
count    12914.000000
mean       359.585836
std        800.742716
min          4.000000
25%         73.000000
50%        158.845000
75%        340.965000
max      19995.000000
Name: price, dtype: float64

=== price_per_unit Stats ===
count    12914.000000
mean       333.491706
std        791.933775
min          0.060000
25%         59.940000
50%        135.000000
75%        310.000000
max      19995.000000
Name: price_per_unit, dtype: float64

=== Date ===
['2026-05-04' '2026-05-25']

=== Sample ===
          sku  source category  sub_category product_type                                       